<a href="https://colab.research.google.com/github/athishsreeram/tubenotebook/blob/main/YoutubeVideoUrl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependency (run once)
!pip install yt-dlp


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 16.9 MB/s eta 0:00:00


In [3]:
import yt_dlp
import json
import csv
from datetime import datetime


# ─────────────────────────────
# CONFIG
# ─────────────────────────────

CHANNEL_URL = "https://www.youtube.com/@rajshamani/videos"
MAX_VIDEOS = 100          # fast scan limit
DEEP_FETCH_TOP_N = 10     # deep analysis on top N


# ─────────────────────────────
# STEP 1 — FAST FETCH
# ─────────────────────────────

def get_channel_videos(channel_url, max_videos=None):
    ydl_opts = {
        "quiet": True,
        "extract_flat": True,
        "skip_download": True,
        "ignoreerrors": True,
        "playlistend": max_videos,
    }

    videos = []

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        entries = info.get("entries", [])

        for entry in entries:
            if not entry:
                continue

            video_id = entry.get("id")
            url = f"https://www.youtube.com/watch?v={video_id}"

            videos.append({
                "title": entry.get("title"),
                "url": url,
                "video_id": video_id,
                "view_count": entry.get("view_count") or 0,
                "duration": entry.get("duration"),
                "upload_date": entry.get("upload_date"),
            })

    return videos


# ─────────────────────────────
# STEP 2 — SCORE & PICK TOP
# ─────────────────────────────

def rank_videos(videos):
    return sorted(
        videos,
        key=lambda x: (x["view_count"] or 0),
        reverse=True
    )


# ─────────────────────────────
# STEP 3 — DEEP FETCH
# ─────────────────────────────

def enrich_video(video_url):
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)

        return {
            "description": info.get("description"),
            "like_count": info.get("like_count"),
            "comment_count": info.get("comment_count"),
            "tags": info.get("tags"),
            "categories": info.get("categories"),
            "chapters": info.get("chapters"),
            "thumbnail": info.get("thumbnail"),
            "uploader": info.get("uploader"),
            "channel_id": info.get("channel_id"),
            "subtitles": list(info.get("subtitles", {}).keys()),
            "auto_captions": list(info.get("automatic_captions", {}).keys()),
        }


# ─────────────────────────────
# STEP 4 — MERGE DATA
# ─────────────────────────────

def enrich_top_videos(videos, top_n):
    for v in videos[:top_n]:
        print(f"🔍 Deep fetching: {v['title'][:50]}...")
        try:
            extra = enrich_video(v["url"])
            v.update(extra)

            # engagement score
            views = v.get("view_count") or 1
            likes = v.get("like_count") or 0
            comments = v.get("comment_count") or 0
            v["engagement_score"] = round((likes + comments) / views, 5)

        except Exception as e:
            print(f"❌ Failed: {e}")

    return videos


# ─────────────────────────────
# STEP 5 — SAVE
# ─────────────────────────────

def save_to_json(data):
    filename = f"youtube_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"💾 Saved JSON: {filename}")


def save_to_csv(data):
    filename = f"youtube_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

    # flatten for CSV
    keys = set()
    for d in data:
        keys.update(d.keys())

    keys = list(keys)

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)

    print(f"💾 Saved CSV: {filename}")


# ─────────────────────────────
# RUN PIPELINE
# ─────────────────────────────

print("🚀 Step 1: Fetching videos...")
videos = get_channel_videos(CHANNEL_URL, MAX_VIDEOS)

print(f"✅ Found {len(videos)} videos")

print("\n📊 Step 2: Ranking...")
videos = rank_videos(videos)

print(f"\n🔥 Step 3: Deep analysis on top {DEEP_FETCH_TOP_N} videos...")
videos = enrich_top_videos(videos, DEEP_FETCH_TOP_N)

print("\n💾 Step 4: Saving...")
save_to_json(videos)
save_to_csv(videos)

print("\n🎯 DONE — You now have a content intelligence dataset")

🚀 Step 1: Fetching videos...
✅ Found 100 videos

📊 Step 2: Ranking...

🔥 Step 3: Deep analysis on top 10 videos...
🔍 Deep fetching: President of France on Trump, India, Modi, Tech & ...


🔍 Deep fetching: Billionaire’s Brain vs Your Brain: Morning Routine...


🔍 Deep fetching: Bharti Singh On Trolls, Body Shaming, Comedy, Fami...


🔍 Deep fetching: Life of S*x Workers: Prostitution, Human Trafficki...


🔍 Deep fetching: Sunita Williams On 286 Days in Space, NASA Mission...


...

🔍 Deep fetching: Vikas Divyakirti on India, China, Power Struggles,...


🔍 Deep fetching: Yuzi Chahal On Divorce, Friends, Cricket, S*icidal...


🔍 Deep fetching: Karan Aujla on Parents, Canada, Love, Loneliness, ...


🔍 Deep fetching: Khan Sir on World War 3, India vs Pakistan, China,...



💾 Step 4: Saving...
💾 Saved JSON: youtube_data_20260321_020920.json
💾 Saved CSV: youtube_data_20260321_020920.csv

🎯 DONE — You now have a content intelligence dataset
